# 8.7 - GraphRAG & Knowledge Graphs

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook works through GraphRAG hands-on: A knowledge graph from scratch: triples + BFS; Neo4j: the production graph database; Hybrid GraphRAG: vectors + graph together; Microsoft GraphRAG: Leiden communities, four search modes; LightRAG: cheap indexing, incremental updates; LlamaIndex PropertyGraphIndex: schema-guided KG.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Uncomment and run this once on a fresh machine or in Colab to install everything the notebook touches - the Gemini SDK, the graph databases, and the graph-RAG frameworks. On a machine that already has them, skip it.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install numpy llama-index llama-index-llms-google-genai llama-index-graph-stores-neo4j neo4j lightrag-hku transformers torch google-genai python-dotenv -q  # noqa


**Explanation**

A single commented `pip install` line, not code that runs by default. It pulls the whole stack this lesson exercises across its ten steps so later cells import cleanly.

**How the code works - step by step**
- **`google-genai`** - the current Gemini SDK used by almost every cell (extraction, embeddings, answering).
- **`neo4j`** - Python driver for the Neo4j graph database (step 2).
- **`llama-index` + `llama-index-graph-stores-neo4j` + `llama-index-llms-google-genai`** - PropertyGraphIndex and its Neo4j store (step 6).
- **`lightrag-hku`** - the LightRAG framework (step 5).
- **`transformers` + `torch`** - run the IndicNER model locally (step 10).
- **`numpy`** - cosine math for the vector and entity-resolution steps.
- **`python-dotenv`** - load keys from a `.env` file.

**In one line:** one install line for the full graph-RAG stack.

**What you'll see:** (no output - environment prep)

## Setup - API key

Load your Google AI Studio key into the environment so every later `genai.Client()` picks it up automatically. Any one provider key is enough to start; nothing here is hardcoded.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

Environment prep, not a model call. It sets `GOOGLE_API_KEY` only if it is not already present, so the client constructor in later cells can read it without you passing it around.

**How the code works - step by step**
- **`import os`** - access the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the variable without clobbering a real key you already exported; the empty default is a placeholder you replace with your own key.

**In one line:** make the Gemini key available to every cell that calls `genai.Client()`.

**What you'll see:** (no output - environment prep)

## 1 - A knowledge graph from scratch: triples + BFS

GraphRAG's entire idea fits in three operations you can write by hand: an LLM extracts `(subject, relation, object)` triples, a breadth-first search walks the edges outward, and the LLM answers from what the walk gathered. Building it once without a framework makes every later tool legible - they are all industrial versions of this.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`)

In [ ]:
# KNOWLEDGE GRAPH FROM SCRATCH - extract triples with an LLM, traverse with BFS
# pip install google-genai
import json
from collections import defaultdict
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from the environment

class SimpleKnowledgeGraph:
    def __init__(self):
        self.triples = []
        self.adjacency = defaultdict(list)

    def extract_from_text(self, text):
        prompt = ("Extract entity-relationship triples as a JSON array of "
                  "{subject, relation, object}. Keep entities short (1-3 words).\n\n"
                  f"Text: {text}\n\nReturn ONLY the JSON array:")
        raw = client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()
        if raw.startswith("```"): raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        try:
            for t in json.loads(raw):
                tri = (t["subject"], t["relation"], t["object"])
                self.triples.append(tri)
                self.adjacency[tri[0].lower()].append(tri)
                self.adjacency[tri[2].lower()].append(tri)
            return len(self.triples)
        except Exception:
            return len(self.triples)

    def get_neighbors(self, entity, hops=2):
        visited, result, queue = set(), [], [(entity.lower(), 0)]
        while queue:
            node, depth = queue.pop(0)
            if node in visited or depth > hops: continue
            visited.add(node)
            for tri in self.adjacency.get(node, []):
                result.append(tri)
                for e in (tri[0].lower(), tri[2].lower()):
                    if e not in visited: queue.append((e, depth + 1))
        return result

    def query(self, question):
        ent = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"List the key entities in this question as a JSON array of strings.\nQ: {question}").text.strip()
        if ent.startswith("```"): ent = ent.split("\n", 1)[1].rsplit("```", 1)[0]
        try: entities = json.loads(ent)
        except Exception: entities = question.lower().split()[:3]
        ctx = set()
        for e in entities:
            for s, r, o in self.get_neighbors(e, hops=2):
                ctx.add(f"{s} --[{r}]--> {o}")
        ans = client.models.generate_content(model="gemini-2.5-flash",
            contents="Answer using ONLY this knowledge graph.\nGraph:\n" + "\n".join(ctx) + f"\n\nQ: {question}\nA:").text.strip()
        return {"answer": ans, "triples_used": len(ctx)}

kg = SimpleKnowledgeGraph()
for t in [
    "Netsetos offers the GenAI course, which requires Python.",
    "The GenAI course has Module 8, which covers RAG.",
    "Module 8 (RAG) is taught by Instructor A.",
]:
    print(f"total triples: {kg.extract_from_text(t)}")
print(kg.query("Who teaches the module that covers RAG?"))

# Output:
# total triples: 2
# total triples: 4
# total triples: 5
# {'answer': 'Instructor A teaches Module 8, which covers RAG.', 'triples_used': 5}  # exact counts vary by LLM extraction

**Explanation**

One small class that IS GraphRAG in miniature. `extract_from_text` turns sentences into triples and indexes them in an adjacency map; `get_neighbors` traverses that map; `query` extracts the question's entities, traverses from them, and answers only from the traversed triples. Read it as: extract a web of edges, then follow the edges to a fact no single sentence states.

**How the code works - step by step**
- **`__init__`** - holds a flat `triples` list plus an `adjacency` dict mapping each entity (lowercased) to the triples it appears in.
- **`extract_from_text`** - prompts `gemini-2.5-flash` for a JSON array of `{subject, relation, object}`, strips any code fence, then stores each triple under BOTH its subject and object so traversal can enter from either end.
- **`get_neighbors`** - a classic BFS: a queue of `(node, depth)` pairs, a `visited` set, fanning out one hop at a time until `depth > hops`, collecting every triple it passes.
- **`query`** - first asks the LLM for the question's key entities, then gathers their 2-hop neighbourhood, formats each triple as `subject --[relation]--> object`, and asks the LLM to answer using ONLY that graph text.
- **Driver** - ingests three sentences (printing the running triple count) then asks a two-hop question: "Who teaches the module that covers RAG?"

**In one line:** extract triples -> index both endpoints -> BFS from the question's entities -> answer from the walk.

**What you'll see:** The triple counter climbs (`2`, `4`, `5`) as each sentence is ingested, then a dict answer like `{'answer': 'Instructor A teaches Module 8, which covers RAG.', 'triples_used': 5}` - the exact counts vary because extraction is an LLM call.

## 2 - Neo4j: the production graph database

A Python dict of triples is fine for a demo; production needs a real graph database. Neo4j stores nodes and relationships natively and queries them with Cypher - SQL for graphs. You `MERGE` to insert without duplicates and `MATCH` a path pattern to traverse, and an LLM can even write the Cypher from a plain-English question.

> **Needs a running Neo4j** (the `docker run` line in the comment) and a Gemini key to execute the writes.

In [ ]:
# NEO4J GRAPHRAG - production knowledge-graph storage
# pip install neo4j google-genai
# docker run -p 7474:7474 -p 7687:7687 -e NEO4J_AUTH=neo4j/password neo4j
from neo4j import GraphDatabase
from google import genai

client = genai.Client()

class Neo4jGraphRAG:
    def __init__(self, uri="bolt://localhost:7687", user="neo4j", password="password"):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def add_triple(self, subject, relation, obj):
        with self.driver.session() as s:
            s.run("MERGE (a:Entity {name:$subj}) "
                  "MERGE (b:Entity {name:$obj}) "
                  "MERGE (a)-[:REL {type:$rel}]->(b)",
                  subj=subject, obj=obj, rel=relation)

    def neighbours(self, entity, hops=2):
        with self.driver.session() as s:
            rows = s.run(f"MATCH p=(a:Entity {{name:$name}})-[*1..{hops}]->(b) "
                         "RETURN a.name AS src, last(relationships(p)).type AS rel, b.name AS dst",
                         name=entity)
            return [(r["src"], r["rel"], r["dst"]) for r in rows]

    def text_to_cypher(self, question):
        prompt = ("Convert the question to a Neo4j Cypher query. Schema: "
                  "(:Entity {name})-[:REL {type}]->(:Entity). Return ONLY the Cypher.\n\n"
                  f"Q: {question}\nCypher:")
        return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip().strip("`")

# The pattern (needs a running Neo4j to execute the writes):
print("MERGE triples -> MATCH a path -> RETURN the nodes -> LLM answers\n")
for q, cypher in [
    ("What does Netsetos offer?",
     "MATCH (:Entity {name:'Netsetos'})-[:REL]->(c) RETURN c.name"),
    ("Multi-hop: what topics are in the course Netsetos offers?",
     "MATCH (:Entity {name:'Netsetos'})-[:REL]->(c)-[:REL {type:'HAS_MODULE'}]->(m) RETURN m.name"),
]:
    print(f"Q: {q}\nCypher: {cypher}\n")

# Output:
# MERGE triples -> MATCH a path -> RETURN the nodes -> LLM answers
# Q: What does Netsetos offer?  Cypher: MATCH (:Entity {name:'Netsetos'})-[:REL]->(c) RETURN c.name
# Q: Multi-hop...              Cypher: MATCH (:Entity {name:'Netsetos'})-[:REL]->(c)-[:REL {type:'HAS_MODULE'}]->(m) RETURN m.name

**Explanation**

A wrapper class over the Neo4j driver plus a text-to-Cypher helper. The class shows the same extract/traverse pattern as step 1, but backed by a database that scales; the runnable part at the bottom just prints the pattern and two example Cypher queries (it does not need a live database).

**How the code works - step by step**
- **`__init__`** - opens a Neo4j driver against `bolt://localhost:7687`.
- **`add_triple`** - `MERGE`s both entity nodes and the `:REL` edge between them; `MERGE` is idempotent, so re-running the same insert never creates a duplicate.
- **`neighbours`** - runs a variable-length path `MATCH p=(a)-[*1..hops]->(b)`, the Cypher way to traverse N hops, returning source, the last relationship's type, and destination.
- **`text_to_cypher`** - hands the LLM the schema and a question and asks it to emit only a Cypher query.
- **Driver** - prints the four-stage pattern, then two `(question, hand-written Cypher)` pairs showing a one-hop and a two-hop `MATCH`.

**In one line:** MERGE nodes/edges idempotently, MATCH a variable-length path to traverse, let the LLM write the Cypher.

**What you'll see:** The line `MERGE triples -> MATCH a path -> RETURN the nodes -> LLM answers`, followed by the two example questions each paired with its Cypher (a one-hop `Netsetos -> course` and a two-hop `Netsetos -> course -> module`).

## 3 - Hybrid GraphRAG: vectors + graph together

The best production systems do not choose vector or graph - they run both and hand the LLM both contexts. Vector search finds chunks that resemble the query (broad, fuzzy); graph traversal chains related entities (precise, multi-hop). Hybrid gives the model the similar chunks AND the relationship chain.

> **Needs a Gemini API key** for embeddings and generation.

In [ ]:
# HYBRID GRAPHRAG - vector breadth + graph relationship-chains
# pip install google-genai numpy
import numpy as np
from google import genai
# from step 1: SimpleKnowledgeGraph

client = genai.Client()

def embed(text):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

class HybridGraphRAG:
    def __init__(self):
        self.chunks, self.embs = [], []
        self.kg = SimpleKnowledgeGraph()   # from step 1

    def ingest(self, text):
        self.chunks.append(text); self.embs.append(embed(text))
        self.kg.extract_from_text(text)

    def query(self, question):
        qe = embed(question)
        sims = sorted(((i, float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e))))
                       for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        vector_ctx = "\n".join(self.chunks[i] for i, _ in sims[:3])   # SIMILAR
        graph_ctx = self.kg.query(question)["answer"]                # CONNECTED
        return client.models.generate_content(model="gemini-2.5-flash",
            contents=(f"Answer using BOTH.\nVector (similar chunks):\n{vector_ctx}\n\n"
                      f"Graph (relationship chains):\n{graph_ctx}\n\nQ: {question}\nA:")).text.strip()

print("Vector finds SIMILAR chunks; graph finds CONNECTED entities; hybrid uses each where it wins.")

# Output:
# Vector finds SIMILAR chunks; graph finds CONNECTED entities; hybrid uses each where it wins.

**Explanation**

A class that fuses step 1's `SimpleKnowledgeGraph` with a plain cosine vector store. `ingest` feeds each document to both; `query` runs both retrievers and asks the LLM to synthesize one answer from the two contexts. It is a wiring pattern, not new retrieval math - the runnable line just states the principle.

**How the code works - step by step**
- **`embed`** - one Gemini `embed_content` call with `gemini-embedding-001`, returned as a NumPy vector.
- **`__init__`** - keeps parallel `chunks`/`embs` lists plus an embedded `SimpleKnowledgeGraph`.
- **`ingest`** - stores the chunk, its embedding, AND extracts triples into the graph, so both retrievers see every document.
- **`query`** - embeds the question, cosine-ranks the chunks and takes the top 3 (`vector_ctx`, the SIMILAR half), calls `self.kg.query(...)` for the graph answer (`graph_ctx`, the CONNECTED half), then prompts the LLM to answer using BOTH.
- **Driver** - prints the one-line summary of what each retriever is for.

**In one line:** cosine-rank chunks for breadth, traverse the graph for the chain, let the LLM merge both.

**What you'll see:** A single printed line: `Vector finds SIMILAR chunks; graph finds CONNECTED entities; hybrid uses each where it wins.` (the class itself needs ingested docs and API calls to produce a real answer).

## 4 - Microsoft GraphRAG: Leiden communities, four search modes

Microsoft GraphRAG's killer feature is community summaries. After extracting the graph it runs Leiden community detection to cluster related entities, then has an LLM summarize each cluster - which unlocks "what are the main themes across ALL documents?" via Global Search's map-reduce over those summaries. It is CLI-first because entity extraction dominates indexing cost.

In [ ]:
# MICROSOFT GRAPHRAG - the CLI pipeline (Leiden communities + 4 search modes)
# pip install graphrag
#
# 1) Initialize a workspace (writes settings.yaml + .env)
#    graphrag init --root ./ragtest
#
# 2) Point settings.yaml at your model, then index. --method fast = NLP extraction (cheaper)
#    graphrag index --root ./ragtest --method fast
#
# 3) Query with one of four search methods (default is global)
#    graphrag query --root ./ragtest --method local  "Who teaches the RAG module?"
#    graphrag query --root ./ragtest --method global "What are the main themes?"
#    graphrag query --root ./ragtest --method drift  "How does RAG relate to fine-tuning?"

phases = [
    "1. Chunk text into ~1,200-token units",
    "2. LLM extracts entities + relationships + claims  (the bulk of the cost)",
    "3. Leiden community detection clusters the graph (hierarchical)",
    "4. LLM writes a natural-language summary for each community",
    "5. Link communities back to their source text units",
    "6. Embed entities + community reports into a vector store",
]
for p in phases:
    print(p)
print("\nGlobal Search then map-reduces over the community summaries at query time.")

# Output:
# 1. Chunk text into ~1,200-token units
# 2. LLM extracts entities + relationships + claims  (the bulk of the cost)
# ... (6 phases) ...
# Global Search then map-reduces over the community summaries at query time.

**Explanation**

A reference cell, not a live call: the CLI workflow is in the comments and the code just prints the six indexing phases. Read it to understand what `graphrag init/index/query` does under the hood and where the cost lives.

**How the code works - step by step**
- **Comment block** - the three CLI steps: `graphrag init` (writes config), `graphrag index --method fast` (NLP extraction instead of an LLM, far cheaper), and `graphrag query --method [local|global|drift]`.
- **`phases` list** - the six indexing stages: chunk -> LLM extracts entities/relations/claims (the cost) -> Leiden clustering -> LLM community summaries -> link communities to source text -> embed into a vector store.
- **Loop** - prints each phase, then a note that Global Search map-reduces over the community summaries at query time.

**In one line:** index once into Leiden communities with LLM summaries, then Local/Global/DRIFT/Basic search chooses how to read them.

**What you'll see:** The six numbered pipeline phases printed in order, followed by `Global Search then map-reduces over the community summaries at query time.`

## 5 - LightRAG: cheap indexing, incremental updates

Full GraphRAG front-loads cost into indexing; the 2026 alternatives move that cost around. LightRAG skips community pre-compute for dual-level (entity + topic) retrieval and updates incrementally. The choice is about your data: stable and queried often, or changing and cost-sensitive.

> **Needs a Gemini API key** and writes a local `./lightrag_store`.

In [ ]:
# LIGHTRAG - dual-level retrieval, cheap indexing, incremental updates
# pip install lightrag-hku google-genai numpy
import asyncio
import numpy as np
from google import genai
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

client = genai.Client()

# Adapt Gemini to LightRAG's function interfaces
async def gemini_complete(prompt, system_prompt=None, history_messages=[], **kwargs):
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text

def gemini_embed(texts):
    r = client.models.embed_content(model="gemini-embedding-001", contents=texts)
    return np.array([e.values for e in r.embeddings])

async def main():
    rag = LightRAG(
        working_dir="./lightrag_store",
        llm_model_func=gemini_complete,
        embedding_func=EmbeddingFunc(embedding_dim=3072, max_token_size=8192, func=gemini_embed),
    )
    await rag.initialize_storages()
    await initialize_pipeline_status()

    await rag.ainsert("Netsetos offers a GenAI course; Module 8 covers RAG and GraphRAG.")
    for mode in ["naive", "local", "global", "hybrid"]:
        ans = await rag.aquery("What does Module 8 cover?", param=QueryParam(mode=mode))
        print(f"[{mode:7s}] {ans[:56]}")

asyncio.run(main())

# Output:
# [naive  ] top-k chunks, no graph
# [local  ] entity-level neighbourhood
# [global ] topic-level themes
# [hybrid ] both levels merged  (entity + topic)

**Explanation**

Shows how to drop Gemini into a third-party framework by adapting its completion and embedding functions, then run the same insert/query shape as any RAG - but with four retrieval modes exposed by one API. The async structure is required by LightRAG's interface.

**How the code works - step by step**
- **`gemini_complete` / `gemini_embed`** - thin adapters matching LightRAG's expected function signatures, wrapping Gemini generation and embeddings.
- **`LightRAG(...)`** - wired with those functions plus an `EmbeddingFunc` declaring the 3072-dim embedding space.
- **`initialize_storages` + `initialize_pipeline_status`** - required async setup before use.
- **`ainsert`** - builds the dual-level graph from one document (supports incremental adds, no full reindex).
- **Mode loop** - runs the same question through `naive`, `local`, `global`, and `hybrid` `QueryParam` modes and prints a truncated answer for each.
- **`asyncio.run(main())`** - drives the coroutine.

**In one line:** adapt Gemini into LightRAG, insert once, then query at four retrieval levels.

**What you'll see:** Four bracketed lines, one per mode - `naive` (top-k chunks, no graph), `local` (entity neighbourhood), `global` (topic themes), `hybrid` (both merged); the comment shows the shape, real text depends on the model.

## 6 - LlamaIndex PropertyGraphIndex: schema-guided KG

PropertyGraphIndex is LlamaIndex's graph path and it replaced the deprecated KnowledgeGraphIndex. Instead of bare triples it stores LABELED nodes (PERSON, COURSE) with properties and rich edges, and `SchemaLLMPathExtractor` constrains extraction to an ontology you define - with `strict=True` the LLM can only emit types you allow, keeping the graph clean.

> **Needs a running Neo4j and a Gemini API key.**

In [ ]:
# LLAMAINDEX PROPERTYGRAPHINDEX - schema-guided KG (replaces KnowledgeGraphIndex)
# pip install llama-index llama-index-graph-stores-neo4j llama-index-llms-google-genai
from typing import Literal
from llama_index.core import PropertyGraphIndex, Document, Settings
from llama_index.core.indices.property_graph import SchemaLLMPathExtractor
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.llms.google_genai import GoogleGenAI

Settings.llm = GoogleGenAI(model="gemini-2.5-flash")

# Constrain extraction to an ontology - strict=True keeps the graph clean
entities = Literal["COMPANY", "COURSE", "TOPIC", "PERSON"]
relations = Literal["OFFERS", "COVERS", "REQUIRES", "TAUGHT_BY"]
schema = {"COMPANY": ["OFFERS"], "COURSE": ["COVERS", "REQUIRES"], "TOPIC": ["TAUGHT_BY"]}

extractor = SchemaLLMPathExtractor(
    llm=Settings.llm, possible_entities=entities,
    possible_relations=relations, kg_validation_schema=schema, strict=True,
)

docs = [Document(text="Netsetos offers the GenAI course, which covers RAG and requires Python.")]
index = PropertyGraphIndex.from_documents(
    docs, kg_extractors=[extractor],
    property_graph_store=Neo4jPropertyGraphStore(username="neo4j", password="password", url="bolt://localhost:7687"),
)
retriever = index.as_retriever(include_text=True, similarity_top_k=2)
print(retriever.retrieve("What does the GenAI course cover?"))

# Output: a list of NodeWithScore over the matching labeled subgraph, e.g.
# (Netsetos:COMPANY)-[OFFERS]->(GenAI course:COURSE)-[COVERS]->(RAG:TOPIC)

**Explanation**

A build-and-retrieve example over a labeled property graph backed by Neo4j. The core idea is the schema: you declare the allowed entity types, relation types, and which relations each entity may have, and the strict extractor refuses anything off-ontology.

**How the code works - step by step**
- **`Settings.llm = GoogleGenAI(...)`** - sets Gemini as the extraction/query model.
- **`entities` / `relations` / `schema`** - the ontology: allowed node types, edge types, and a map of which relations each entity type can originate.
- **`SchemaLLMPathExtractor(..., strict=True)`** - forces the LLM's extraction to conform; off-schema facts are dropped rather than added.
- **`PropertyGraphIndex.from_documents`** - extracts from the document into a `Neo4jPropertyGraphStore`.
- **`index.as_retriever(...).retrieve(...)`** - queries the labeled subgraph, returning scored nodes with their source text.

**In one line:** declare an ontology, extract strictly against it, retrieve over a labeled property graph.

**What you'll see:** A list of `NodeWithScore` objects over the matching labeled subgraph; the comment illustrates the shape as `(Netsetos:COMPANY)-[OFFERS]->(GenAI course:COURSE)-[COVERS]->(RAG:TOPIC)`.

## 7 - Graph construction: entity resolution, the hidden bottleneck

Every downstream retrieval problem traces back to construction, and the underrated decision is entity resolution: merging "GenAI Course" and "the GenAI course" into one node. Unresolved duplicates silently split the graph so traversals miss links. This cell shows the embedding-similarity first pass that flags candidate duplicates.

> **Needs a Gemini API key** for embeddings.

In [ ]:
# GRAPH CONSTRUCTION - entity resolution (the underrated bottleneck)
# pip install google-genai numpy
import numpy as np
from google import genai

client = genai.Client()

def embed(t):
    return np.array(client.models.embed_content(model="gemini-embedding-001", contents=t).embeddings[0].values)

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Merge duplicate mentions before they fragment the graph
mentions = ["GenAI Course", "the GenAI course", "GenAI programme", "Data Science Course"]
vecs = [embed(m) for m in mentions]
for i in range(len(mentions)):
    for j in range(i + 1, len(mentions)):
        if cos(vecs[i], vecs[j]) > 0.88:          # candidate duplicate -> LLM verifies in prod
            print(f"likely same entity: '{mentions[i]}' == '{mentions[j]}'")

# Output:
# likely same entity: 'GenAI Course' == 'the GenAI course'
# likely same entity: 'GenAI Course' == 'GenAI programme'

**Explanation**

A tiny measurement harness, not a full pipeline: it embeds several surface forms of entity names and flags high-cosine pairs as likely-the-same. In production an LLM would verify the ambiguous pairs before merging; here the threshold does the flagging.

**How the code works - step by step**
- **`embed`** - one `gemini-embedding-001` call per mention, returned as a NumPy vector.
- **`cos`** - plain cosine similarity between two vectors.
- **`mentions`** - four names, three of which are variants of the same course plus one genuinely different course.
- **Double loop** - compares every pair; any with cosine `> 0.88` is printed as a candidate duplicate (the point where an LLM verifier would step in for real).

**In one line:** embed name variants, and high cosine similarity flags duplicates before they fragment the graph.

**What you'll see:** Two "likely same entity" lines merging `GenAI Course` with `the GenAI course` and with `GenAI programme`; `Data Science Course` stays distinct (below threshold).

## 8 - The hybrid router: graph this query, or not?

In production GraphRAG is not a default - it is a routed choice. On simple facts vector RAG is faster, cheaper, and often more accurate; on multi-hop and thematic queries the graph wins. So the highest-leverage pattern is a classifier that sends each query to the retriever that wins it.

> **Needs a Gemini API key.**

In [ ]:
# HYBRID ROUTER - send each query to the retriever that wins it
# pip install google-genai
from google import genai

client = genai.Client()

def route(question):
    prompt = ("Classify the question as 'factual' (a single fact / lookup) or "
              "'relational' (multi-hop, cross-document, or 'main themes'). One word.\n\n"
              f"Q: {question}\nClass:")
    kind = client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip().lower()
    return "vector_rag" if "factual" in kind else "graph_rag"

for q in ["What is the refund policy?",
          "What are the main themes across all support tickets?",
          "Who teaches the module that covers the topic in this complaint?"]:
    print(f"{route(q):10s} <- {q}")

# Output:
# vector_rag <- What is the refund policy?
# graph_rag  <- What are the main themes across all support tickets?
# graph_rag  <- Who teaches the module that covers the topic in this complaint?

**Explanation**

The single most important production decision in the lesson, in one function. `route` asks the LLM to label a question factual or relational and maps that to `vector_rag` or `graph_rag`. The router call is cheap relative to running the wrong retriever - or both.

**How the code works - step by step**
- **`route`** - prompts `gemini-2.5-flash` to classify the question as `factual` (single lookup) or `relational` (multi-hop / cross-document / themes) in one word, then returns `vector_rag` if the answer contains "factual" else `graph_rag`.
- **Driver loop** - runs three questions through it: a refund-policy lookup, a corpus-wide themes question, and a multi-hop instructor question.

**In one line:** classify each query once, then send facts to vector RAG and relational/thematic queries to graph RAG.

**What you'll see:** Three routed lines: `vector_rag <- What is the refund policy?`, `graph_rag <- ...main themes...`, and `graph_rag <- Who teaches the module...` - the fact goes to vectors, the multi-hop and thematic ones to the graph.

## 9 - LLM-as-judge: comprehensiveness and diversity

The order-aware metrics from 8.4 (NDCG, MRR) score whether the right chunk ranked high - but GraphRAG's value is comprehensiveness and diversity, which those metrics cannot see. Microsoft's framework judges four dimensions with an LLM via pairwise comparison.

> **Needs a Gemini API key.**

In [ ]:
# LLM-AS-JUDGE - GraphRAG needs comprehensiveness/diversity, not NDCG/MRR
# pip install google-genai
from google import genai

client = genai.Client()

def judge(question, answer_a, answer_b):
    prompt = ("Compare two answers on comprehensiveness, diversity, empowerment, and "
              "directness. Say which is better overall (A or B) and why in one line.\n\n"
              f"Q: {question}\n\nA: {answer_a}\n\nB: {answer_b}\n\nVerdict:")
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()

print(judge(
    "What are the main themes across the course?",
    "The course covers RAG.",                                              # narrow (vector-ish)
    "Themes: retrieval systems, model architecture, prompting, and production ops.",  # broad (graph-ish)
))

# Output:
# B is more comprehensive: it surfaces cross-document themes, not a single chunk's fact.

**Explanation**

A pairwise LLM-as-judge in one function. `judge` hands the model a question and two candidate answers and asks which is better on comprehensiveness, diversity, empowerment, and directness - the kind of holistic ruling a rank metric cannot produce.

**How the code works - step by step**
- **`judge`** - prompts `gemini-2.5-flash` to compare answers A and B on the four dimensions and name the better one with a one-line reason.
- **Driver** - calls it on a themes question with a deliberately narrow answer (`The course covers RAG.`) versus a broad one listing several themes.

**In one line:** show the judge two answers, ask which is more comprehensive/diverse, not which ranked first.

> Randomize A/B order and average several runs in production - the judge has position and length bias.

**What you'll see:** A one-line verdict picking the broad answer (B) as more comprehensive because it surfaces cross-document themes rather than a single chunk's fact; wording varies per run.

## 10 - Multilingual + India GraphRAG: Indic NER

Graph construction in Hindi and specialized domains needs specialist extractors, not a generic LLM. Hindi has no capitalization to signal names, free word order, and rich morphology, so AI4Bharat's IndicNER (and MuRIL) beat a generic model - and for Indian legal graphs there is a whole domain stack.

> **Needs `transformers` + `torch`** - downloads and runs the IndicNER model locally (a GPU helps but is not required).

In [ ]:
# MULTILINGUAL + INDIA GRAPHRAG - domain/Indic NER beats a generic LLM here
# pip install transformers torch
from transformers import pipeline

# IndicNER handles Hindi entities where a generic model has no capitalization cues
ner = pipeline("token-classification", model="ai4bharat/IndicNER", aggregation_strategy="simple")
text = "नेटसेटोस हैदराबाद में जेनएआई कोर्स प्रदान करता है।"   # Netsetos offers a GenAI course in Hyderabad
for ent in ner(text):
    print(ent["word"], "->", ent["entity_group"])

# For Indian legal graphs: OpenNyAI (Court, Judge, Statute, Provision) + InLegalBERT
# embeddings + NyayGraph (IPC knowledge graph on Neo4j).

# Output:
# नेटसेटोस -> ORG
# हैदराबाद -> LOC

**Explanation**

A local Hugging Face inference call, not a Gemini API call - this is a domain model running on your machine. It tags entities in a Devanagari sentence where a generic English NER has no capitalization cues to work from; the comment points at the legal-domain equivalents.

**How the code works - step by step**
- **`pipeline("token-classification", model="ai4bharat/IndicNER", aggregation_strategy="simple")`** - loads IndicNER and groups sub-word tokens into whole entities.
- **`text`** - a Hindi sentence (Netsetos offers a GenAI course in Hyderabad).
- **Loop** - prints each detected entity word with its group (ORG / LOC / PER).
- **Comment** - the legal stack for Indian graphs: OpenNyAI (Court/Judge/Statute/Provision), InLegalBERT embeddings, NyayGraph (IPC graph on Neo4j).

**In one line:** a domain-trained Indic NER tags Devanagari entities a generic model would miss.

**What you'll see:** Two tagged lines - `नेटसेटोस -> ORG` and `हैदराबाद -> LOC` - the organization and location pulled from Hindi text with no capitalization to cue them.

Across ten runnable cells you built GraphRAG from its three primitives - extract (subject, relation, object) triples, traverse them (BFS in a dict, then a variable-length Cypher path in Neo4j), and answer from the walk - then hybridized it with vector search and toured the 2026 tooling: Microsoft GraphRAG's Leiden communities and four search modes, the cheaper LightRAG/LazyGraphRAG/HippoRAG line, LlamaIndex's schema-constrained PropertyGraphIndex, the entity-resolution bottleneck, a per-query router, LLM-as-judge evaluation, and the Indic/legal NER stack. The through-line: vectors find similar, graphs find connected, and production discipline is routing each query to the one that wins it. Next, Lesson 8.10 turns the router into an agent that decides when and how deep to traverse, and 8.11 turns the LLM judge into a CI eval gate.